# 01 — Análisis Exploratorio: Salarios y Empleo en ZMG

**Proyecto:** GDL Ecosystem Intelligence  
**Fuente:** ENOE (INEGI) — Encuesta Nacional de Ocupación y Empleo  
**Período:** 2018 T1 — 2024 T4  

### Objetivo
Analizar la distribución salarial en la Zona Metropolitana de Guadalajara, comparando trabajadores del sector tech vs. el resto de la economía, y su evolución en el tiempo.

### Pregunta central
> ¿Ha generado el boom del nearshoring en Jalisco una brecha salarial significativa entre el sector tech y el resto de la economía local?


In [ ]:
import os
from pathlib import Path

# Ruta fija a la raíz del proyecto
project_root = Path(r"C:\Users\emmys\OneDrive\Documents\GDL-ECO-INT")
os.chdir(project_root)

print("Directorio actual:", Path.cwd())
print("Existe data/raw/enoe:", Path("data/raw/enoe").exists())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Agregar raíz del proyecto al path
sys.path.insert(0, str(Path.cwd()))

from src.ingestion.enoe_loader import load_multiple_quarters, summary_stats

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette(['#1D9E75', '#D85A30', '#378ADD', '#BA7517'])

# Ruta correcta desde la raíz del proyecto
FIGURES_DIR = Path("reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Setup completo ✓')
print('FIGURES_DIR:', FIGURES_DIR.resolve())

## 1. Carga de datos

In [ ]:
# Cargar todos los trimestres disponibles
# Datos descargados: 2023 T4, 2024 T1-T4, 2025 T1-T4

df = load_multiple_quarters(start_year=2023, end_year=2025)

if df.empty:
    print('⚠️  No hay datos cargados. Revisar docs/fuentes.md')
else:
    print(f'Registros totales: {len(df):,}')
    print(f'Períodos disponibles: {df["periodo"].nunique()}')
    print(f'\nMuestra del dataset:')
    display(df.head())

## 2. Perfil general del mercado laboral ZMG

In [ ]:
# Distribución de trabajadores por municipio
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Trabajadores por municipio
    mun_counts = df.groupby('municipio_nombre').size().sort_values(ascending=True)
    mun_counts.plot(kind='barh', ax=axes[0], color='#1D9E75')
    axes[0].set_title('Trabajadores encuestados por municipio (ENOE)', fontsize=12)
    axes[0].set_xlabel('Registros')

    # Tech vs No-Tech
    sector_counts = df['es_tech'].value_counts()
    sector_counts.index = ['No-Tech', 'Tech']
    sector_counts.plot(kind='bar', ax=axes[1], color=['#D85A30', '#1D9E75'], rot=0)
    axes[1].set_title('Trabajadores Tech vs No-Tech en ZMG', fontsize=12)
    axes[1].set_ylabel('Registros')

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '01_perfil_laboral_zmg.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura guardada: 01_perfil_laboral_zmg.png')

## 3. Brecha salarial: Tech vs No-Tech

In [ ]:
# Ingreso por hora: distribución comparativa
if not df.empty:
    # Filtrar valores atípicos extremos (top 1%)
    p99 = df['ing_x_hrs'].quantile(0.99)
    df_clean = df[df['ing_x_hrs'] <= p99].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribución de salarios
    for sector, color, label in [(True, '#1D9E75', 'Tech'), (False, '#D85A30', 'No-Tech')]:
        subset = df_clean[df_clean['es_tech'] == sector]['ing_x_hrs'].dropna()
        axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=label, density=True)

    axes[0].set_title('Distribución de ingreso por hora (MXN)', fontsize=12)
    axes[0].set_xlabel('Ingreso por hora (MXN)')
    axes[0].set_ylabel('Densidad')
    axes[0].legend()

    # Boxplot por sector
    df_clean['Sector'] = df_clean['es_tech'].map({True: 'Tech', False: 'No-Tech'})
    df_clean.boxplot(column='ing_x_hrs', by='Sector', ax=axes[1],
                    boxprops=dict(color='#1D9E75'),
                    medianprops=dict(color='#D85A30', linewidth=2))
    axes[1].set_title('Boxplot de ingreso por hora por sector', fontsize=12)
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Ingreso por hora (MXN)')
    plt.suptitle('')

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_brecha_salarial_tech_notech.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Estadísticas descriptivas
    print('\n=== Estadísticas: Ingreso por hora (MXN) ===')
    display(
        df_clean.groupby('Sector')['ing_x_hrs']
        .agg(['mean', 'median', 'std', 'count'])
        .round(2)
        .rename(columns={'mean': 'Media', 'median': 'Mediana', 'std': 'Desv. Est.', 'count': 'N'})
    )

## 4. Evolución de la brecha salarial en el tiempo

In [ ]:
# Brecha salarial mediana por trimestre
if not df.empty:
    stats = summary_stats(df)

    fig, ax = plt.subplots(figsize=(14, 5))

    for sector, color, label in [('Tech', '#1D9E75', 'Sector Tech'), ('No-Tech', '#D85A30', 'Resto economía')]:
        subset = stats[stats['sector'] == sector]
        ax.plot(subset['periodo'], subset['mediana'], marker='o', markersize=4,
               color=color, label=label, linewidth=2)
        ax.fill_between(subset['periodo'], subset['p25'], subset['p75'],
                       alpha=0.15, color=color)

    ax.set_title('Evolución del ingreso mediano por hora (MXN) — ZMG 2018–2024', fontsize=13)
    ax.set_xlabel('Período')
    ax.set_ylabel('Ingreso mediano por hora (MXN)')
    ax.legend()

    # Rotar etiquetas del eje X
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '03_evolucion_brecha_salarial.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Hallazgos preliminares

*(Completar al terminar el análisis con los hallazgos reales)*

| Métrica | Valor |
|---|---|
| Ingreso mediano/hora sector tech (2024) | ___ MXN |
| Ingreso mediano/hora sector no-tech (2024) | ___ MXN |
| Brecha porcentual | __% |
| Crecimiento brecha 2018→2024 | __ pp |

### Próximo notebook
→ `02_eda_denue.ipynb` — Análisis de densidad y distribución geográfica de empresas tech en ZMG
